# BuildMarshalAI Voice Transcription Service

A separate OpenAI Whisper speech-to-text API for the BuildMarshalAI chat interface. This notebook does not load ColPali or Qwen and can run independently from the document backend.

In [ ]:
import os, subprocess, sys
os.environ['PIP_NO_CACHE_DIR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'openai-whisper==20250625', 'fastapi==0.115.6',
    'uvicorn[standard]==0.34.0', 'python-multipart==0.0.20',
    'pyngrok==7.2.2', 'nest-asyncio==1.6.0', 'httpx==0.27.2'
], check=True)
print('Dependencies ready')

In [ ]:
import asyncio, logging, tempfile, time
from pathlib import Path
from typing import Optional

import nest_asyncio
import torch
import whisper
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('BuildMarshalVoice')

MODEL_NAME = 'base'  # multilingual, compact, and suitable for short chat queries
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_DIR = Path('/kaggle/working/whisper_models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MAX_UPLOAD_BYTES = 20 * 1024 * 1024
MAX_AUDIO_SECONDS = 60
ALLOWED_EXTENSIONS = {'.webm', '.wav', '.mp3', '.m4a', '.mp4', '.ogg', '.opus', '.flac', '.aac'}

logger.info(f'Loading OpenAI Whisper {MODEL_NAME} on {DEVICE}...')
WHISPER_MODEL = whisper.load_model(MODEL_NAME, device=DEVICE, download_root=str(MODEL_DIR))
logger.info('Whisper model ready')
TRANSCRIBE_LOCK = asyncio.Lock()

app = FastAPI(title='BuildMarshalAI Voice Service', version='1.0.0')
app.add_middleware(
    CORSMiddleware, allow_origins=['*'], allow_credentials=False,
    allow_methods=['*'], allow_headers=['*']
)

@app.get('/')
async def root():
    return {'service': 'BuildMarshalAI Voice', 'docs': '/docs', 'health': '/api/health'}

@app.get('/api/health')
async def health():
    return {
        'status': 'healthy', 'service': 'voice-transcription',
        'model': f'openai-whisper-{MODEL_NAME}', 'device': DEVICE,
        'cuda_available': torch.cuda.is_available(),
        'max_audio_seconds': MAX_AUDIO_SECONDS
    }

def _transcribe_file(path: str, language: Optional[str]):
    started = time.perf_counter()
    audio = whisper.load_audio(path)
    duration = len(audio) / 16000
    if duration > MAX_AUDIO_SECONDS + 1:
        raise ValueError(f'Audio is {duration:.1f}s; maximum is {MAX_AUDIO_SECONDS}s')
    result = WHISPER_MODEL.transcribe(
        audio, language=language or None, task='transcribe',
        fp16=(DEVICE == 'cuda'), temperature=0,
        condition_on_previous_text=False, verbose=False
    )
    segments = [
        {'start': round(float(s['start']), 2), 'end': round(float(s['end']), 2), 'text': s['text'].strip()}
        for s in result.get('segments', []) if s.get('text', '').strip()
    ]
    return {
        'text': result.get('text', '').strip(),
        'language': result.get('language'),
        'duration_seconds': round(duration, 2),
        'processing_seconds': round(time.perf_counter() - started, 2),
        'segments': segments, 'model': f'openai-whisper-{MODEL_NAME}'
    }

@app.post('/api/transcribe')
async def transcribe_audio(file: UploadFile = File(...), language: Optional[str] = Form(None)):
    suffix = Path(file.filename or 'voice.webm').suffix.lower() or '.webm'
    if suffix not in ALLOWED_EXTENSIONS:
        raise HTTPException(415, detail=f'Unsupported audio type: {suffix}')
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp:
            temp_path = temp.name
            total = 0
            while chunk := await file.read(1024 * 1024):
                total += len(chunk)
                if total > MAX_UPLOAD_BYTES:
                    raise HTTPException(413, detail='Audio file exceeds the 20 MB limit')
                temp.write(chunk)
        if total == 0:
            raise HTTPException(400, detail='The uploaded audio file is empty')
        async with TRANSCRIBE_LOCK:
            try:
                result = await asyncio.to_thread(_transcribe_file, temp_path, language)
            except ValueError as error:
                raise HTTPException(400, detail=str(error)) from error
            except Exception as error:
                logger.exception('Transcription failed')
                raise HTTPException(500, detail=f'Transcription failed: {error}') from error
        return result
    finally:
        await file.close()
        if temp_path:
            Path(temp_path).unlink(missing_ok=True)

print(f'Voice API ready | model={MODEL_NAME} | device={DEVICE} | routes={len(app.routes)}')

In [ ]:
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

NGROK_AUTH_TOKEN = UserSecretsClient().get_secret('NGROK_AUTH_TOKEN')
if not NGROK_AUTH_TOKEN:
    raise RuntimeError('Add and enable NGROK_AUTH_TOKEN in Kaggle Secrets')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()
tunnel = ngrok.connect(8001)
print('\n' + '=' * 64)
print('BuildMarshalAI Voice Service is LIVE')
print('=' * 64)
print(f'VOICE URL : {tunnel.public_url}')
print(f'HEALTH    : {tunnel.public_url}/api/health')
print(f'API DOCS  : {tunnel.public_url}/docs')
print('Keep this cell running and paste VOICE URL into frontend Settings.')
print('=' * 64)
uvicorn.run(app, host='0.0.0.0', port=8001)